In [2]:
%pip install numpy pandas scikit-learn imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 10.7 MB/s  0:00:000.6 MB/s eta 0:00:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 34.7 MB/s  0:00:0036.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 32.5 MB/s  0:00:006.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 15.0 MB/s  0:00:01a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [imbalanced-learn]2m 9/10 [imbalanced-learn]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os

print(os.getcwd())
print(os.listdir())

/Users/nikitathakur/Desktop/Jupyter/Python Study
['sec 6 practice.ipynb', 'demo.txt', 'Practice 1-6.ipynb', 'Crash course.ipynb', 'Obesity_DataSet_2.csv', 'section 6.ipynb', 'obesity FE code.ipynb', 'section 4.ipynb', 'myfile.txt', '.ipynb_checkpoints', 'section 5.ipynb', 'section 3.ipynb']


In [13]:
# ============================================================
# AUTOMATA-BASED FEATURE ENGINEERING FOR OBESITY PREDICTION
# THREE-BRANCH IMPLEMENTATION
#
# Branch I   : Original features + conventional feature selection
# Branch II  : Original features + automata-derived features
# Branch III : Automata-derived + weighted + fuzzy + hybrid features
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import RandomOverSampler


# ============================================================
# 1. SETTINGS
# ============================================================

DATASET_FILE = "Obesity_DataSet_2.csv"

TARGET = "BMI_WHO"

OUTPUT_FOLDER = "outputs"

RANDOM_STATE = 42

# Number of original features retained in Branch I
TOP_K_FEATURES = 15


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(DATASET_FILE)

print("\n========================================")
print("ORIGINAL DATASET")
print("========================================")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    TARGET,
    "Age",
    "TotChol",
    "BPSysAve",
    "PhysActive",
    "Smoke100"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "\nRequired columns are missing:\n"
        + str(missing_columns)
        + "\n\nAvailable columns are:\n"
        + str(df.columns.tolist())
    )


# ============================================================
# 4. COMMON DATA PREPROCESSING
# ============================================================

print("\n========================================")
print("COMMON PREPROCESSING")
print("========================================")

# Remove samples where target class is missing
df = df.dropna(subset=[TARGET]).copy()

# Reset index
df = df.reset_index(drop=True)

print("Shape after preprocessing:", df.shape)

print("\nTarget distribution:")
print(df[TARGET].value_counts())


# ============================================================
# 5. BINARY ENCODING FUNCTION
# ============================================================
# ============================================================
# 5. BINARY ENCODING FUNCTION
# ============================================================

def encode_binary(series):
    """
    Converts Yes/No healthcare variables into binary form.

    Yes -> 1
    No  -> 0
    Missing -> NaN
    """

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "yes": 1.0,
        "no": 0.0,
        "y": 1.0,
        "n": 0.0,
        "true": 1.0,
        "false": 0.0
    }

    return cleaned.map(mapping).astype(float)


# ============================================================
# 6. AUTOMATA-DERIVED FEATURE EXTRACTION
# ============================================================
def extract_automata_features(data):

    Z = pd.DataFrame(index=data.index)

    # --------------------------------------------------------
    # z1 - AGE STATE
    # --------------------------------------------------------

    Z["Age_State"] = np.select(
        [
            data["Age"] < 30,
            (data["Age"] >= 30) & (data["Age"] < 50),
            data["Age"] >= 50
        ],
        [0, 1, 2],
        default=np.nan
    )

    # --------------------------------------------------------
    # z2 - CHOLESTEROL STATE
    # --------------------------------------------------------

    Z["Cholesterol_State"] = np.select(
        [
            data["TotChol"] < 5.17,
            (data["TotChol"] >= 5.17) & (data["TotChol"] < 6.21),
            data["TotChol"] >= 6.21
        ],
        [0, 1, 2],
        default=np.nan
    )

    # --------------------------------------------------------
    # z3 - BLOOD PRESSURE STATE
    # --------------------------------------------------------

    Z["BP_State"] = np.select(
        [
            data["BPSysAve"] < 120,
            (data["BPSysAve"] >= 120) & (data["BPSysAve"] < 140),
            data["BPSysAve"] >= 140
        ],
        [0, 1, 2],
        default=np.nan
    )

    # --------------------------------------------------------
    # z4 - PHYSICAL ACTIVITY
    # --------------------------------------------------------

    Z["PhysActive_Binary"] = encode_binary(
        data["PhysActive"]
    )

    # --------------------------------------------------------
    # z5 - SMOKING
    # --------------------------------------------------------

    Z["Smoke_Binary"] = encode_binary(
        data["Smoke100"]
    )

    # --------------------------------------------------------
    # z6 - AGE × ACTIVITY INTERACTION
    # --------------------------------------------------------

    Z["Age_Activity_Interaction"] = (
        data["Age"] * Z["PhysActive_Binary"]
    )

    return Z


# ============================================================
# 7. EXTRACT AUTOMATA FEATURES FROM SAME ORIGINAL DATASET
# ============================================================

Z = extract_automata_features(df)

# Fill any accidental missing automata value
for column in Z.columns:

    if Z[column].isna().any():

        Z[column] = Z[column].fillna(
            Z[column].median()
        )


print("\n========================================")
print("AUTOMATA-DERIVED FEATURES")
print("========================================")

print(Z.head())

print("\nDerived feature shape:", Z.shape)


# ============================================================
# 8. TARGET
# ============================================================

y = df[TARGET].copy()

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print("\nTarget classes:")

for i, class_name in enumerate(
    target_encoder.classes_
):

    print(i, "=", class_name)


# ============================================================
# HELPER FUNCTION:
# PREPARE ORIGINAL VARIABLES FOR ML
# ============================================================

def prepare_original_features(data):

    X = data.drop(
        columns=[TARGET]
    ).copy()

    # --------------------------------------------------------
    # Handle numeric missing values
    # --------------------------------------------------------

    numeric_columns = (
        X.select_dtypes(
            include=np.number
        ).columns
    )

    for column in numeric_columns:

        X[column] = X[column].fillna(
            X[column].median()
        )


    # --------------------------------------------------------
    # Handle categorical missing values
    # --------------------------------------------------------

    categorical_columns = (
        X.select_dtypes(
            exclude=np.number
        ).columns
    )

    for column in categorical_columns:

        if X[column].isna().any():

            mode_value = X[column].mode()

            if len(mode_value) > 0:

                X[column] = X[column].fillna(
                    mode_value.iloc[0]
                )

            else:

                X[column] = X[column].fillna(
                    "Unknown"
                )


    # --------------------------------------------------------
    # One-hot encoding
    # --------------------------------------------------------

    X = pd.get_dummies(
        X,
        drop_first=False,
        dtype=int
    )

    return X


# ============================================================
# PREPARE ORIGINAL FEATURE SPACE
# ============================================================

X_original = prepare_original_features(df)


# ============================================================
# ============================================================
# BRANCH I
# BASELINE CLASSICAL FEATURE ENGINEERING
# ============================================================
# ============================================================

print("\n\n========================================")
print("BRANCH I")
print("BASELINE CLASSICAL FEATURES")
print("========================================")


# Start from ORIGINAL dataset only
X_branch1_all = X_original.copy()


# ------------------------------------------------------------
# MUTUAL INFORMATION FEATURE SELECTION
# ------------------------------------------------------------

mi_scores = mutual_info_classif(
    X_branch1_all,
    y_encoded,
    random_state=RANDOM_STATE
)


mi_table = pd.DataFrame({

    "Feature":
        X_branch1_all.columns,

    "MI_Score":
        mi_scores

})


mi_table = mi_table.sort_values(
    by="MI_Score",
    ascending=False
)


number_to_select = min(
    TOP_K_FEATURES,
    len(mi_table)
)


selected_features = (
    mi_table
    .head(number_to_select)
    ["Feature"]
    .tolist()
)


X_branch1 = (
    X_branch1_all[
        selected_features
    ]
    .copy()
)


print("\nTop selected features:")

print(
    mi_table.head(
        number_to_select
    )
)


print(
    "\nBranch I feature shape:",
    X_branch1.shape
)


# ============================================================
# ============================================================
# BRANCH II
# ORIGINAL + AUTOMATA-MERGED FEATURES
# ============================================================
# ============================================================

print("\n\n========================================")
print("BRANCH II")
print("AUTOMATA-MERGED FEATURES")
print("========================================")


# Branch II starts with the same original feature space
# then appends z1 ... z6.

X_branch2 = pd.concat(
    [

        X_original.reset_index(
            drop=True
        ),

        Z.reset_index(
            drop=True
        )

    ],

    axis=1
)


# Remove duplicate column names if present
X_branch2 = X_branch2.loc[
    :,
    ~X_branch2.columns.duplicated()
]


print(
    "\nOriginal encoded features:",
    X_original.shape[1]
)


print(
    "Automata-derived features:",
    Z.shape[1]
)


print(
    "Final Branch II features:",
    X_branch2.shape[1]
)


print("\nBranch II sample:")

print(
    X_branch2[
        Z.columns
    ].head()
)


# ============================================================
# ============================================================
# BRANCH III
# FULL AUTOMATA-INTEGRATED FEATURE ENGINEERING
# ============================================================
# ============================================================

print("\n\n========================================")
print("BRANCH III")
print("AUTOMATA-INTEGRATED FEATURES")
print("========================================")


# Branch III begins from ONLY automata-derived representation
X_branch3 = Z.copy()

X_branch3 = X_branch3.replace([np.inf, -np.inf], np.nan)

print("Missing values before cleaning:")
print(X_branch3.isna().sum())

print("\nOriginal PhysActive values:")
print(df["PhysActive"].unique())

print("\nOriginal Smoke100 values:")
print(df["Smoke100"].unique())

for col in X_branch3.columns:

    if X_branch3[col].isna().all():
        raise ValueError(
            f"{col} contains only NaN values. "
            "Its original variable must be encoded correctly."
        )

    X_branch3[col] = X_branch3[col].fillna(
        X_branch3[col].median()
    )

print("\nMissing values after cleaning:")
print(X_branch3.isna().sum())
# ============================================================
# 9. WEIGHTED AUTOMATA
# ============================================================

# Estimate relevance of each derived feature using
# Mutual Information against BMI_WHO.

mi_automata = mutual_info_classif(

    X_branch3,

    y_encoded,

    random_state=RANDOM_STATE
)


# Normalize weights
if mi_automata.sum() == 0:

    automata_weights = np.ones(
        len(mi_automata)
    ) / len(mi_automata)

else:

    automata_weights = (
        mi_automata
        /
        mi_automata.sum()
    )


weight_table = pd.DataFrame({

    "Feature":
        X_branch3.columns,

    "Weight":
        automata_weights

})


print("\nAutomata feature weights:")

print(weight_table)


# ------------------------------------------------------------
# Weighted Automata Score
#
# SW = Σ wi zi
# ------------------------------------------------------------

base_automata_columns = [

    "Age_State",

    "Cholesterol_State",

    "BP_State",

    "PhysActive_Binary",

    "Smoke_Binary",

    "Age_Activity_Interaction"
]


# Normalize automata-derived features before weighted integration
Z_weighted = X_branch3[base_automata_columns].copy()

for col in Z_weighted.columns:
    min_val = Z_weighted[col].min()
    max_val = Z_weighted[col].max()

    if max_val != min_val:
        Z_weighted[col] = (
            Z_weighted[col] - min_val
        ) / (
            max_val - min_val
        )
    else:
        Z_weighted[col] = 0.0

# Weighted Automata Score
X_branch3["Weighted_Automata_Score"] = np.dot(
    Z_weighted,
    automata_weights
)


# ============================================================
# 10. FUZZY AUTOMATA MEMBERSHIP FUNCTIONS
# ============================================================

def fuzzy_high_bp(bp):

    if pd.isna(bp):
        return np.nan

    if bp <= 130:
        return 0.0

    elif bp < 140:
        return (
            bp - 130
        ) / 10.0

    else:
        return 1.0


def fuzzy_high_cholesterol(chol):

    if pd.isna(chol):
        return np.nan

    if chol <= 200:
        return 0.0

    elif chol < 240:

        return (
            chol - 200
        ) / 40.0

    else:
        return 1.0


def fuzzy_senior_age(age):

    if pd.isna(age):
        return np.nan

    if age <= 40:
        return 0.0

    elif age < 60:

        return (
            age - 40
        ) / 20.0

    else:
        return 1.0


# ============================================================
# 11. COMPUTE FUZZY MEMBERSHIP VALUES
# ============================================================

mu_age = (
    df["Age"]
    .apply(
        fuzzy_senior_age
    )
    .fillna(0)
    .to_numpy()
)


mu_bp = (
    df["BPSysAve"]
    .apply(
        fuzzy_high_bp
    )
    .fillna(0)
    .to_numpy()
)


mu_cholesterol = (
    df["TotChol"]
    .apply(
        fuzzy_high_cholesterol
    )
    .fillna(0)
    .to_numpy()
)


# Inactivity membership
mu_inactive = (

    1
    -
    X_branch3[
        "PhysActive_Binary"
    ].to_numpy()

)


# Smoking membership
mu_smoking = (

    X_branch3[
        "Smoke_Binary"
    ].to_numpy()

)


# ============================================================
# 12. FUZZY RULES
# ============================================================

# ------------------------------------------------------------
# Rule 1
# IF Age is Senior
# AND BP is High
# AND Physical Activity is Inactive
# THEN Risk is High
# ------------------------------------------------------------

alpha_1 = np.minimum.reduce(

    [
        mu_age,
        mu_bp,
        mu_inactive
    ]

)


# ------------------------------------------------------------
# Rule 2
# IF Cholesterol is High
# AND Smoker
# THEN Risk is High
# ------------------------------------------------------------

alpha_2 = np.minimum(

    mu_cholesterol,

    mu_smoking

)


# ------------------------------------------------------------
# Rule 3
# IF Age is Senior
# AND Cholesterol is High
# THEN Risk is High
# ------------------------------------------------------------

alpha_3 = np.minimum(

    mu_age,

    mu_cholesterol

)


# ------------------------------------------------------------
# Rule 4
# IF BP is High
# AND Smoker
# THEN Risk is High
# ------------------------------------------------------------

alpha_4 = np.minimum(

    mu_bp,

    mu_smoking

)


# ============================================================
# 13. FUZZY AUTOMATA SCORE
# ============================================================

# Equal rule weights in this implementation

lambda_1 = 1.0
lambda_2 = 1.0
lambda_3 = 1.0
lambda_4 = 1.0


fuzzy_score = (

    lambda_1 * alpha_1
    +
    lambda_2 * alpha_2
    +
    lambda_3 * alpha_3
    +
    lambda_4 * alpha_4

) / (

    lambda_1
    +
    lambda_2
    +
    lambda_3
    +
    lambda_4

)


X_branch3[
    "Fuzzy_Automata_Score"
] = fuzzy_score


# ============================================================
# 14. NORMALIZATION FOR HYBRID INTEGRATION
# ============================================================

def min_max_normalize(series):

    minimum = series.min()

    maximum = series.max()

    if maximum == minimum:

        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (

        series - minimum

    ) / (

        maximum - minimum

    )


weighted_normalized = min_max_normalize(

    X_branch3[
        "Weighted_Automata_Score"
    ]

)


interaction_normalized = min_max_normalize(

    X_branch3[
        "Age_Activity_Interaction"
    ]

)


# ============================================================
# 15. HYBRID AUTOMATA INTEGRATION
# ============================================================

# Equal contribution initially.
#
# These coefficients can later be tuned experimentally.

eta_1 = 1 / 3
eta_2 = 1 / 3
eta_3 = 1 / 3


X_branch3[
    "Hybrid_Automata_Score"
] = (

    eta_1
    *
    weighted_normalized

    +

    eta_2
    *
    X_branch3[
        "Fuzzy_Automata_Score"
    ]

    +

    eta_3
    *
    interaction_normalized

)


print("\nFinal Branch III features:")

print(
    X_branch3.columns.tolist()
)


print(
    "\nBranch III shape:",
    X_branch3.shape
)


print("\nBranch III sample:")

print(
    X_branch3.head()
)


# ============================================================
# 16. CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


# ============================================================
# 17. SAVE UNBALANCED FEATURE SPACES
# ============================================================

branch1_dataset = X_branch1.copy()

branch1_dataset[TARGET] = (
    y.reset_index(drop=True)
)


branch2_dataset = X_branch2.copy()

branch2_dataset[TARGET] = (
    y.reset_index(drop=True)
)


branch3_dataset = X_branch3.copy()

branch3_dataset[TARGET] = (
    y.reset_index(drop=True)
)


branch1_dataset.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "branch1_baseline_features.csv"
    ),

    index=False
)


branch2_dataset.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "branch2_automata_merged_features.csv"
    ),

    index=False
)


branch3_dataset.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "branch3_automata_integrated_features.csv"
    ),

    index=False
)


# Save MI tables

mi_table.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "branch1_feature_importance.csv"
    ),

    index=False
)


weight_table.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "automata_feature_weights.csv"
    ),

    index=False
)


# ============================================================
# 18. OPTIONAL BALANCED EXPORTS
# ============================================================

def create_balanced_dataset(
    X,
    y,
    file_name
):

    ros = RandomOverSampler(
        random_state=RANDOM_STATE
    )

    X_balanced, y_balanced = (
        ros.fit_resample(
            X,
            y
        )
    )


    balanced = X_balanced.copy()

    balanced[TARGET] = y_balanced


    balanced.to_csv(

        os.path.join(
            OUTPUT_FOLDER,
            file_name
        ),

        index=False
    )


    print(
        file_name,
        "shape:",
        balanced.shape
    )


print("\n========================================")
print("CREATING BALANCED EXPORTS")
print("========================================")


create_balanced_dataset(

    X_branch1,
    y,

    "branch1_baseline_balanced.csv"
)


create_balanced_dataset(

    X_branch2,
    y,

    "branch2_automata_merged_balanced.csv"
)


create_balanced_dataset(

    X_branch3,
    y,

    "branch3_automata_integrated_balanced.csv"
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n========================================")
print("FEATURE ENGINEERING COMPLETED")
print("========================================")


print(
    "Branch I:",
    X_branch1.shape
)


print(
    "Branch II:",
    X_branch2.shape
)


print(
    "Branch III:",
    X_branch3.shape
)


print(
    "\nGenerated files are available in:",
    OUTPUT_FOLDER
)


ORIGINAL DATASET
Shape: (7481, 16)

Columns:
['BMI_WHO', 'Age', 'Gender', 'Race1', 'Education', 'HHIncome', 'PhysActive', 'Smoke100', 'Diabetes', 'BPSysAve', 'TotChol', 'Alcohol12PlusYr', 'MaritalStatus', 'Work', 'Height', 'Depressed']

COMMON PREPROCESSING
Shape after preprocessing: (7384, 16)

Target distribution:
BMI_WHO
Obese          2623
OverWeight     2440
NormWeight     2174
UnderWeight     147
Name: count, dtype: int64

AUTOMATA-DERIVED FEATURES
   Age_State  Cholesterol_State  BP_State  PhysActive_Binary  Smoke_Binary  \
0        1.0                0.0       0.0                0.0           1.0   
1        1.0                0.0       0.0                0.0           1.0   
2        1.0                0.0       0.0                0.0           1.0   
3        1.0                2.0       0.0                0.0           1.0   
4        1.0                1.0       0.0                1.0           0.0   

   Age_Activity_Interaction  
0                       0.0  
1          

In [9]:
# ============================================================
# FIX BINARY ENCODING
# ============================================================

def encode_binary(series):

    # Convert values safely to strings
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "yes": 1.0,
        "no": 0.0,
        "y": 1.0,
        "n": 0.0,
        "true": 1.0,
        "false": 0.0
    }

    return cleaned.map(mapping).astype(float)


# Regenerate automata features using corrected encoding
Z = extract_automata_features(df)

# Fill remaining missing values
# Smoke100 has some missing records
for col in Z.columns:

    if Z[col].isna().any():

        Z[col] = Z[col].fillna(
            Z[col].median()
        )


# Recompute interaction after cleaning
Z["Age_Activity_Interaction"] = (
    df["Age"] * Z["PhysActive_Binary"]
)


print("Corrected automata-derived features:")
print(Z.head())

print("\nMissing values:")
print(Z.isna().sum())

Corrected automata-derived features:
   Age_State  Cholesterol_State  BP_State  PhysActive_Binary  Smoke_Binary  \
0        1.0                0.0       0.0                0.0           1.0   
1        1.0                0.0       0.0                0.0           1.0   
2        1.0                0.0       0.0                1.0           0.0   
3        2.0                0.0       0.0                1.0           1.0   
4        2.0                0.0       0.0                1.0           0.0   

   Age_Activity_Interaction  
0                       0.0  
1                       0.0  
2                      45.0  
3                      66.0  
4                      58.0  

Missing values:
Age_State                   0
Cholesterol_State           0
BP_State                    0
PhysActive_Binary           0
Smoke_Binary                0
Age_Activity_Interaction    0
dtype: int64


In [14]:
print(X_branch3["Fuzzy_Automata_Score"].describe())

print(
    "Non-zero fuzzy scores:",
    (X_branch3["Fuzzy_Automata_Score"] > 0).sum()
)

count    7384.000000
mean        0.039814
std         0.109428
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.500000
Name: Fuzzy_Automata_Score, dtype: float64
Non-zero fuzzy scores: 1159
